# Signals and Systems Final Project

### Miguel Posada, James Pitts, and Maddox Nichols

### NOTE:
This code can't work in Google Colab since it doesn't support live audio recording which is fundamental to our project. Download the notebook and run it locally.

The code below includes a setup block and then three separate filters. Each one runs separately so just run setup then pick any filter block to run.

- [Band Pass](#band_pass)
- [Low Pass](#low_pass)
- [High Pass](#high_pass)


[Github with Visualizers](https://github.com/maddox0816/signals-and-systems)


### Import Libraries and Define Parameters

In [1]:
import numpy as np
import sounddevice as sd
import time

#default config stuff that shouldn't need to be changed
fs = 44100          
blocksize = 2048    
input_device = None  
output_device = None 

In [ ]:
import sounddevice as sd; print(sd.query_devices())

<a id='band_pass'></a>
## Band Pass 

Implemented by running the signal through our high pass filter and then taking the ouput of that and running it through the low pass filter.

In [ ]:
f_low = 80    # min frequency (used in high pass)
f_high = 1500   #max frequency (used in low pass)

#find filter coefficients (james method)
a_hp = 1 / (1 + (2 * np.pi * f_low / fs))
a_lp = (2 * np.pi * f_high / fs) / (1 + (2 * np.pi * f_high / fs))

#set up filter buffers
hp_x1, hp_y1 = 0.0, 0.0
hp_x2, hp_y2 = 0.0, 0.0

lp_y1 = 0.0
lp_y2 = 0.0


def apply_standard_bpf(x):
    global hp_x1, hp_y1, hp_x2, hp_y2
    global lp_y1, lp_y2
    
    y = np.zeros_like(x)

    for n in range(len(x)):
        #high pass
        out_hp1 = a_hp * (hp_y1 + x[n] - hp_x1)       
        out_hp2 = a_hp * (hp_y2 + out_hp1 - hp_x2)
        
        hp_x1, hp_y1 = x[n], out_hp1
        hp_x2, hp_y2 = out_hp1, out_hp2

        #then do low pass
        out_lp1 = lp_y1 + a_lp * (out_hp2 - lp_y1)   
        out_lp2 = lp_y2 + a_lp * (out_lp1 - lp_y2)
        
        lp_y1 = out_lp1
        lp_y2 = out_lp2

        #save result
        y[n] = out_lp2

    return y



def callback(indata, outdata, frames, time_info, status):
    if status:
        print(status)

    processed_audio = apply_standard_bpf(indata[:, 0])

    #send to left and right channels
    outdata[:, 0] = processed_audio
    outdata[:, 1] = processed_audio

#run for 10 sec
stream = sd.Stream(channels=(1, 2), samplerate=fs, blocksize=blocksize, callback=callback)
stream.start()
time.sleep(10)
stream.stop()


<a id='low_pass'></a>
## Low Pass Filter

We first run the signal through a band pass to remove extra noise above and below what a guitar string would produce. Then we run it through our low pass filter that is based on the equations derived from the RC circuit.


In [2]:
#low pass cuttoff
f_final_lp = 100




#band pass to stop noise
f_low = 80    
f_high = 400

#find filter coefficients (james method)
a_hp = 1 / (1 + (2 * np.pi * f_low / fs))
a_lp = (2 * np.pi * f_high / fs) / (1 + (2 * np.pi * f_high / fs))
a_final_lp = (2 * np.pi * f_final_lp / fs) / (1 + (2 * np.pi * f_final_lp / fs))

#set up filter buffers
hp_x1, hp_y1 = 0.0, 0.0
hp_x2, hp_y2 = 0.0, 0.0

lp_y1 = 0.0
lp_y2 = 0.0

final_lp_y1 = 0.0
final_lp_y2 = 0.0


def apply_filters(x):
    global hp_x1, hp_y1, hp_x2, hp_y2 
    global lp_y1, lp_y2
    global final_lp_y1, final_lp_y2
    
    y = np.zeros_like(x)

    for n in range(len(x)):
        #high pass
        out_hp1 = a_hp * (hp_y1 + x[n] - hp_x1)
        out_hp2 = a_hp * (hp_y2 + out_hp1 - hp_x2)
        
        hp_x1, hp_y1 = x[n], out_hp1
        hp_x2, hp_y2 = out_hp1, out_hp2

        #low pass
        out_lp1 = lp_y1 + a_lp * (out_hp2 - lp_y1)
        out_lp2 = lp_y2 + a_lp * (out_lp1 - lp_y2)

        lp_y1 = out_lp1
        lp_y2 = out_lp2

        #main low pass
        out_final1 = final_lp_y1 + a_final_lp * (out_lp2 - final_lp_y1)
        out_final2 = final_lp_y2 + a_final_lp * (out_final1 - final_lp_y2)
        
        final_lp_y1 = out_final1
        final_lp_y2 = out_final2

        # Final filtered sample
        y[n] = out_final2

    return y



def callback(indata, outdata, frames, time_info, status):
    if status:
        print(status)


    processed_audio = apply_filters(indata[:, 0])

    #output to left and right
    outdata[:, 0] = processed_audio
    outdata[:, 1] = processed_audio

#run for 10 sec
print(f"Running Enhanced Band-Pass: {f_low}Hz - {f_high}Hz with Final LP at {f_final_lp}Hz")
stream = sd.Stream(channels=(1, 2), samplerate=fs, blocksize=blocksize, callback=callback)
stream.start()
time.sleep(10)
stream.stop()

Running Enhanced Band-Pass: 80Hz - 400Hz with Final LP at 100Hz


<a id='high_pass'></a>
## High Pass Filter

The sound first runs the signal through a band pass to remove any extra noise and then runs it through the high pass. Then we run it through our high pass filter that is based on the equations derived from the RC circuit.

In [ ]:
#high pass frequency
f_final_hp = 500 



#band pass to stop noise
f_bp_low = 200    
f_bp_high = 1200


#find cooeficients with james method
a_bp_hp = 1 / (1 + (2 * np.pi * f_bp_low / fs))
a_bp_lp = (2 * np.pi * f_bp_high / fs) / (1 + (2 * np.pi * f_bp_high / fs))
a_final_hp = 1 / (1 + (2 * np.pi * f_final_hp / fs))


#set up filter buffers
bp_hp_x1, bp_hp_y1 = 0.0, 0.0
bp_hp_x2, bp_hp_y2 = 0.0, 0.0

bp_lp_y1 = 0.0
bp_lp_y2 = 0.0

final_hp_x1, final_hp_y1 = 0.0, 0.0
final_hp_x2, final_hp_y2 = 0.0, 0.0


def apply_chain(x):
    global bp_hp_x1, bp_hp_y1, bp_hp_x2, bp_hp_y2 
    global bp_lp_y1, bp_lp_y2
    global final_hp_x1, final_hp_y1, final_hp_x2, final_hp_y2
    
    y = np.zeros_like(x)

    for n in range(len(x)):
        #high pass
        out_bp_hp1 = a_bp_hp * (bp_hp_y1 + x[n] - bp_hp_x1)
        out_bp_hp2 = a_bp_hp * (bp_hp_y2 + out_bp_hp1 - bp_hp_x2)
        
        bp_hp_x1, bp_hp_y1 = x[n], out_bp_hp1
        bp_hp_x2, bp_hp_y2 = out_bp_hp1, out_bp_hp2

        #low pass
        out_bp_lp1 = bp_lp_y1 + a_bp_lp * (out_bp_hp2 - bp_lp_y1)
        out_bp_lp2 = bp_lp_y2 + a_bp_lp * (out_bp_lp1 - bp_lp_y2)

        bp_lp_y1 = out_bp_lp1
        bp_lp_y2 = out_bp_lp2

        #run main high pass
        out_final_hp1 = a_final_hp * (final_hp_y1 + out_bp_lp2 - final_hp_x1)
        out_final_hp2 = a_final_hp * (final_hp_y2 + out_final_hp1 - final_hp_x2)
        
        final_hp_x1, final_hp_y1 = out_bp_lp2, out_final_hp1
        final_hp_x2, final_hp_y2 = out_final_hp1, out_final_hp2

        #save output
        y[n] = out_final_hp2

    return y


def callback(indata, outdata, frames, time_info, status):
    if status:
        print(status)

    processed_audio = apply_chain(indata[:, 0])

    #send to left and right channels
    outdata[:, 0] = processed_audio
    outdata[:, 1] = processed_audio



#run for 10 sec
stream = sd.Stream(channels=(1, 2), samplerate=fs, blocksize=blocksize, callback=callback)
stream.start()
time.sleep(10)
stream.stop()